# Create Stockholm Water Prize Awards (PRIZE PATTERN, big-page-inline scrape)

Ingest of the **Stockholm Water Prize**, awarded annually since 1991 by the Stockholm Water Foundation (in collaboration with the Royal Swedish Academy of Sciences, with H.M. King Carl XVI Gustaf of Sweden as official patron). The prize honors outstanding achievements in water research, policy, and management. Laureates include David Schindler (1991), Jorg Imberger (1996), Werner Stumm + James J. Morgan (joint 1999), Kader Asmal (2000), WaterAid (organizational 1995), Sandra Postel (2021), Taikan Oki (2024), Günter Blöschl (2025), and Kaveh Madani (2026).

**Awarding body:** Stockholm International Water Institute — `F4320320937` (SE). Note: the prize is now administered by the dedicated Stockholm Water Foundation (the predecessor URL `siwi.org/prizes/stockholmwaterprize/` redirects to `stockholmwaterfoundation.org`); OpenAlex's funder registry still indexes the institution as SIWI, so we use that funder_id.

**Source:** `stockholmwaterfoundation.org/stockholm-water-prize/laureates` — the foundation's own site (WordPress, no auth, no aggregator). Method #5 on the runbook ladder (static-HTML scrape), specifically the **"big-page-inline" variant**: the entire 36-card laureate corpus is server-side-rendered into one ~1.3MB HTML response — no FacetWP, no admin-ajax, no pagination. First ingest on the project that tests this pattern at scale.

**Corpus** (verified 2026-05-21 full local scrape): **36 laureate rows** spanning **1991-2026** (annual, single recipient with 3 joint-laureate years and 4 organizational recipients):
- 32 individual laureates + 4 organizations (WaterAid 1995, Centre for Science and Environment 2005, International Water Management Institute 2012, Department of Environmental Engineering at TU Denmark 1992)
- 3 joint years (1999 Stumm+Morgan, 2010 Colwell+Johns Hopkins, 2018 Rittmann+van Loosdrecht)
- 13 distinct countries (USA 8, India 4, South Africa 3, Japan/Canada/UK/Denmark 2 each, …)
- 100% citation + country coverage; 35/36 landing-page URL coverage (the 2026 announcement card has no detail page yet)

**Schema:**
- `funder_award_id` = `stockholm-water-prize-{year}-{slug}` where slug is preferentially from the URL (`/laureates/{year}-{slug}-{country}/`) else from the cleaned name. Verified unique.
- `lead_investigator` is PERSON-LEVEL for the 32 individual recipients (given_name + family_name parsed from the country-stripped name; honorific prefixes Dr/Prof/Sir/Dame removed). For the 4 organizational recipients, `given_name` + `family_name` are NULL and `affiliation.name` is the organization name.
- For 3 joint laureates, `is_joint=True` and the first recipient becomes the `lead_investigator`; the co-laureate is captured in the description text.
- `country` comes preferentially from the `<h3>Name, Country</h3>` comma-tail (modern cards) and falls back to URL-slug parsing (older cards like `1991-professor-david-w-schindler-canada`). 100% populated.
- `funder_scheme = 'Stockholm Water Prize'`. Single program.
- `funding_type = 'prize'`.
- `declined` always FALSE.

**Amount = NULL (§6.7 waiver):**
The Stockholm Water Foundation's own site does NOT publish the prize's monetary value on the public landing page, laureates index, or any individual laureate detail page (verified by text-search 2026-05-21). This matches the Wolf Prize (#47), Fields Medal (#50), and Gairdner (#62) precedent — we apply the runbook §6.7 amount waiver and ship `amount = NULL, currency = NULL`. Widely-reported third-party sources place the prize at USD 150,000 historically, but per source-authority discipline we don't ship aggregator-sourced amounts when the awarding body itself declines to publish them. §6.7 amount-coverage is therefore explicitly WAIVED for this funder.

**Source authority** — `stockholmwaterfoundation.org` is the foundation's own site (operationally affiliated with SIWI / the Royal Swedish Academy of Sciences). Method #5 (static-HTML scrape) on the runbook ladder.

**Prerequisites**: Run `scripts/local/stockholm_water_prize_to_s3.py` first to fetch the single big listing page (~10 sec wall-clock), parse 36 cards inline, write parquet, upload to S3.

**Data source**: `https://stockholmwaterfoundation.org/stockholm-water-prize/laureates`

**S3 location**: `s3a://openalex-ingest/awards/stockholm_water_prize/stockholm_water_prize_laureates.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.stockholm_water_prize_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/stockholm_water_prize/stockholm_water_prize_laureates.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.stockholm_water_prize_raw;

## Step 1.5: Inspect raw + money/currency scan + §6.7 NULL-amount waiver acknowledgment

All source columns from the big-page-inline scrape. **Verified per-row coverage on the full extraction** (36 laureate rows, validated 2026-05-21):
- 100% `funder_award_id`, `year`, `name`, `description` (citation)
- 100% `country`
- 97% `landing_page_url` (35/36 — 2026 cohort detail page not yet published)
- `amount` and `currency` are NULL by design — §6.7 waiver, foundation doesn't publish per-laureate monetary amount
- 36 distinct years (1991-2026)
- 32 individuals + 4 organizations
- 3 joint laureates (`is_joint = True`)

Source columns: `funder_award_id`, `year`, `name`, `given_name`, `family_name`, `country`, `recipient_kind`, `is_joint`, `citation`, `display_name`, `description`, `amount` (NULL), `currency` (NULL), `start_date`, `end_date`, `landing_page_url`, `slug_from_url`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.stockholm_water_prize_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.stockholm_water_prize_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 money-shape scan — confirms amount is uniformly NULL by design.
-- This is the §6.7 waiver in action: the awarding body does not publish
-- per-laureate prize amounts on their own site, so we don't fabricate one.
SELECT
    COUNT(amount) AS non_null_amount,
    COUNT(*) AS total_rows,
    COUNT(currency) AS non_null_currency,
    collect_set(currency) AS currencies
FROM openalex.awards.stockholm_water_prize_raw;

In [ ]:
%sql
-- Year + recipient-kind distribution — confirms 1991-2026 span,
-- 32/4 individual/org split, joint-laureate years.
SELECT TRY_CAST(year AS INT) AS year_int,
       recipient_kind,
       is_joint,
       COUNT(*) AS rows
FROM openalex.awards.stockholm_water_prize_raw
GROUP BY year_int, recipient_kind, is_joint
ORDER BY year_int DESC;

## Step 1.6: Fail-fast — verify SIWI funder row exists

In [ ]:
%sql
-- Must return exactly 1 row.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320320937;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.stockholm_water_prize_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320937  -- Stockholm International Water Institute
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,        -- NULL by design (§6.7)
    n.currency,                                     -- NULL by design (§6.7)
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'prize' AS funding_type,
    'Stockholm Water Prize' AS funder_scheme,
    'stockholm_water_prize' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date,   'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date,   1, 4) AS INT) AS end_year,
    -- lead_investigator: PERSON-LEVEL for individuals (given+family from
    -- the script's country-stripped name splitter), ORG-LEVEL for the 4
    -- organizational recipients (NULL person fields, affiliation.name =
    -- the org's own name). For joint laureates, captures the FIRST
    -- recipient only (the co-laureate appears in the description).
    CASE
        WHEN n.recipient_kind = 'individual' AND n.name IS NOT NULL THEN struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                CAST(NULL AS STRING) AS name,
                n.country AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        WHEN n.recipient_kind = 'organization' AND n.name IS NOT NULL THEN struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.name AS name,
                n.country AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    n.declined,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.stockholm_water_prize_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.name IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 100

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'stockholm_water_prize' AND priority = 100;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    100 AS priority  -- Stockholm Water Prize priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.stockholm_water_prize_awards;

## Step 6: Verification

Full §6.1-6.7. **Amount-coverage is EXPLICITLY WAIVED per §6.7** — the foundation does not publish per-laureate amounts on its own site, matching the Wolf Prize / Fields Medal / Gairdner precedent. Verified expectations from the 2026-05-21 extraction:
- Row count: **36** (full roster across 1991-2026)
- `pct_amount = 0%` (by design — §6.7 waiver)
- `currencies = []`
- 1 distinct `funder_scheme` ('Stockholm Water Prize'), 1 distinct `funding_type` ('prize')
- 36 distinct years (1991-2026)
- 32 individual + 4 organizational laureates
- 13 distinct countries (USA 8, India 4, South Africa 3, …)
- `declined = FALSE` everywhere

In [ ]:
%sql
SELECT COUNT(*) AS total_swp_award_rows FROM openalex.awards.stockholm_water_prize_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.stockholm_water_prize_awards;

In [ ]:
%sql
-- §6.3 Data completeness — amount + currency intentionally 0%
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) AS pct_title,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) AS pct_dates,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) AS pct_description
FROM openalex.awards.stockholm_water_prize_awards;

In [ ]:
%sql
-- §6.7 amount + currency — EXPLICITLY WAIVED for this funder.
-- Expected: amount/currency 100% NULL because the Stockholm Water
-- Foundation does not publish per-laureate amounts on its own site.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.stockholm_water_prize_awards;

In [ ]:
%sql
-- Country distribution
SELECT lead_investigator.affiliation.country AS country,
       COUNT(*) AS rows
FROM openalex.awards.stockholm_water_prize_awards
GROUP BY country
ORDER BY rows DESC, country;

In [ ]:
%sql
-- Individuals vs organizations
SELECT
    CASE WHEN lead_investigator.given_name IS NULL
         AND lead_investigator IS NOT NULL THEN 'organization'
         WHEN lead_investigator.given_name IS NOT NULL THEN 'individual'
         ELSE 'unknown' END AS recipient_kind,
    COUNT(*) AS rows
FROM openalex.awards.stockholm_water_prize_awards
GROUP BY recipient_kind;

In [ ]:
%sql
-- Sample recent + earliest laureates
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       lead_investigator.given_name AS given,
       lead_investigator.family_name AS family,
       lead_investigator.affiliation.country AS country,
       start_year
FROM openalex.awards.stockholm_water_prize_awards
WHERE start_year >= 2024 OR start_year <= 1995
ORDER BY start_year DESC
LIMIT 12;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.stockholm_water_prize_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;

In [ ]:
%sql
-- Declined must be FALSE everywhere.
SELECT declined, COUNT(*) AS rows
FROM openalex.awards.stockholm_water_prize_awards
GROUP BY declined;